<a href="https://colab.research.google.com/github/shr1-live/fde-learning-tracker-api-personal/blob/main/docs/notebooks/week2-day3-vector-search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2, Day 3: vector search, step by step

This notebook shows the **same ideas as the real project** (`vector-lab` in the API repository) in small steps you can run and change.
The real project keeps the vectors in Postgres + pgvector. Here they sit in a plain list in memory, so you can see every step.

**What is the same as the real code:** the 12 documents and 25 questions, the chunk splitter (400 characters, 60 overlap), the embedding model (BGE small, 384 numbers),
the query prefix, the filter check, and the merge rule for hybrid search (reciprocal rank fusion).
**What is simplified:** the database (a list instead of Postgres), the HNSW index (we compare with every chunk), and keyword search (word counting instead of Postgres full text).

Run the cells from the top, one at a time (Shift+Enter). Each step has a short explanation and something to try.

## Step 0: install the tools (about 1 minute)

In [1]:
!pip install -q fastembed langchain-text-splitters numpy

## Step 1: the documents and the questions
This cell writes the **real** `corpus.py` from the project: 12 invented company documents (with `region` and `year` labels) and 25 fixed questions.
Some documents are near-duplicates on purpose (UK leave 2025, UK leave 2024 archived, Germany leave), so a filter matters.

In [2]:
%%writefile corpus.py
"""The approved document set (an invented company handbook) and the fixed question set for Week 2, Day 3.

Unlike the 4-document sample used on Days 1 and 2, this set has near-duplicate policies that differ by region and year
(annual leave in the UK and in Germany, an archived 2024 leave policy, hotel caps in Europe and the US). Without a metadata
filter those questions have no single right answer, which is the point of Day 3.
"""
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True)
class Doc:
    doc_id: str
    title: str
    department: str
    region: str  # "All", "UK", "Germany", "US"
    year: int
    body: str


@dataclass(frozen=True)
class Q:
    question: str
    kind: str  # plain | paraphrase | filtered | out_of_scope
    docs: tuple[str, ...] = ()  # doc_ids that count as a correct source; empty = not in the documents
    phrase: str | None = None  # text a correct passage must contain
    filter: dict = field(default_factory=dict)  # metadata filter that a careful user would apply


DOCS: list[Doc] = [
    Doc("leave-uk-2025", "Leave policy (UK)", "hr", "UK", 2025,
        "Every full-time UK employee gets 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month. "
        "Up to 10 unused days can be carried over to the next year; anything above that expires on 31 March.\n\n"
        "Sick leave is separate from annual leave: 12 days per year. A medical certificate is needed when sick leave lasts more than 2 consecutive days. "
        "Tell your manager before 10:00 on the first day of absence.\n\n"
        "Parental leave is 26 weeks for the primary caregiver and 4 weeks for the other parent. Requests must be sent to HR at least 8 weeks before the expected start date."),
    Doc("leave-uk-2024", "Leave policy (UK, archived 2024)", "hr", "UK", 2024,
        "This policy is archived and was replaced in 2025. UK employees received 20 days of paid annual leave per calendar year. "
        "Up to 5 unused days could be carried over and the rest expired on 31 January.\n\n"
        "Sick leave was 10 days per year and a medical certificate was needed after 3 consecutive days of absence."),
    Doc("leave-de-2025", "Leave policy (Germany)", "hr", "Germany", 2025,
        "Every full-time employee in Germany gets 30 days of paid annual leave per calendar year, in line with the works agreement. "
        "Up to 5 unused days can be carried over; anything above that expires on 31 March.\n\n"
        "Sick leave is paid in full for up to 6 weeks. A medical certificate is needed from the fourth day of absence. Tell your manager before 09:00 on the first day.\n\n"
        "Public holidays follow the federal state of the office. Bridge days are announced by the people team in January."),
    Doc("expense-all-2025", "Expense policy", "finance", "All", 2025,
        "Business expenses are reimbursed within 14 days of a complete claim. Submit claims in the finance portal with the original receipt attached. Claims without receipts are rejected.\n\n"
        "Flights must be booked in economy class at least 14 days ahead. Hotel stays are capped at 120 euros per night in most cities and 180 euros in London. Meals during travel are covered up to 45 euros per day.\n\n"
        "Any single purchase above 500 euros needs written approval from a director before you buy. Software subscriptions are bought only by the IT team, never on a personal card."),
    Doc("travel-us-2025", "Travel policy (United States)", "finance", "US", 2025,
        "Employees travelling within the United States book flights in economy class at least 21 days ahead. Hotel stays are capped at 250 dollars per night in New York and San Francisco and 170 dollars elsewhere.\n\n"
        "Meals during travel are covered up to 75 dollars per day. Rental cars need approval from the team lead and must be the smallest suitable class. Claims are paid within 30 days."),
    Doc("remote-all-2025", "Remote work and security", "it", "All", 2025,
        "Employees may work remotely up to 3 days a week after completing their probation period of 3 months. Team days in the office are Tuesday and Thursday.\n\n"
        "Company laptops must use full disk encryption and lock automatically after 5 minutes. Use the VPN on any public Wi-Fi. Never store customer data on personal devices or in personal cloud drives.\n\n"
        "Report a lost device or a suspected phishing email to the security desk within 1 hour. Passwords must have at least 14 characters and multi-factor authentication is mandatory for all company accounts."),
    Doc("remote-de-2025", "Remote work addendum (Germany)", "it", "Germany", 2025,
        "Under the works agreement employees in Germany may work remotely up to 2 days a week. A home-office allowance of 40 euros per month is paid with the salary.\n\n"
        "Working time is recorded in the time-tracking tool and rest periods of 11 hours between working days are respected. Equipment such as a monitor can be ordered once through IT."),
    Doc("onboarding-all-2025", "Onboarding and training", "hr", "All", 2025,
        "New starters get a buddy for the first 8 weeks. The first week covers tools, security training and a tour of the main systems. A check-in with the manager happens at 30, 60 and 90 days.\n\n"
        "Each employee has a learning budget of 1,500 euros per year for courses, books and conferences. Conference attendance needs manager approval at least 30 days before the event. The learning budget does not roll over.\n\n"
        "Mentoring is open to everyone: ask the people team to match you with a mentor from another department. Mentoring pairs meet once every 2 weeks for 45 minutes."),
    Doc("incident-all-2025", "Security incident response", "it", "All", 2025,
        "A security incident is any lost device, leaked credential, suspected phishing email or unauthorised access. Report it to the security desk within 1 hour of noticing it, by phone or through the incident form.\n\n"
        "The security desk assigns a severity from 1 to 4 and a responder. Severity 1 incidents are reviewed by the leadership team within 24 hours. A written summary is shared within 5 working days."),
    Doc("procurement-all-2025", "Procurement and vendors", "finance", "All", 2025,
        "Vendor contracts above 5,000 euros need 3 written quotes and approval from the finance director. Contracts above 50,000 euros also need a security review and legal sign-off.\n\n"
        "Preferred vendors are listed in the finance portal. New vendors must complete the supplier questionnaire before the first purchase order is issued."),
    Doc("performance-all-2025", "Performance reviews", "hr", "All", 2025,
        "Performance reviews take place twice a year, in June and in December. Each review combines a self-assessment, manager feedback and a rating from 1 to 5.\n\n"
        "Pay changes are decided once a year after the December review. An employee who disagrees with a rating can ask for a second review by the next-level manager within 10 working days."),
    Doc("holidays-uk-2026", "Office holidays 2026 (UK)", "hr", "UK", 2026,
        "UK offices are closed on 1 January, 3 April, 6 April, 4 May, 25 May, 31 August, 25 December and 28 December 2026. The office closes at 14:00 on 24 December.\n\n"
        "Staff who work on a bank holiday receive a day off in lieu within 3 months."),
]

QUESTIONS: list[Q] = [
    # Plain: the question uses the words of one document and has one right source.
    Q("How soon are expense claims reimbursed?", "plain", ("expense-all-2025",), "within 14 days"),
    Q("After how many minutes does a company laptop lock itself?", "plain", ("remote-all-2025",), "5 minutes"),
    Q("How long must a password be?", "plain", ("remote-all-2025",), "14 characters"),
    Q("What is the yearly learning budget?", "plain", ("onboarding-all-2025",), "1,500 euros"),
    Q("How often do mentoring pairs meet?", "plain", ("onboarding-all-2025",), "once every 2 weeks"),
    Q("How many written quotes does a vendor contract above 5,000 euros need?", "plain", ("procurement-all-2025",), "3 written quotes"),
    Q("How often are performance reviews held?", "plain", ("performance-all-2025",), "twice a year"),
    Q("How long does a new starter have a buddy?", "plain", ("onboarding-all-2025",), "8 weeks"),
    Q("Within how long must a phishing email be reported?", "plain", ("remote-all-2025", "incident-all-2025"), "within 1 hour"),
    Q("How is the severity of a security incident rated?", "plain", ("incident-all-2025",), "severity from 1 to 4"),
    # Paraphrase: other words for the same fact (keyword search is expected to struggle).
    Q("Can I work from home?", "paraphrase", ("remote-all-2025", "remote-de-2025"), "remotely"),
    Q("Who signs off on a purchase above 500 euros?", "paraphrase", ("expense-all-2025",), "director"),
    Q("What do I need to do before buying something expensive?", "paraphrase", ("expense-all-2025", "procurement-all-2025"), "approval"),
    Q("When can I appeal my performance score?", "paraphrase", ("performance-all-2025",), "second review"),
    Q("What does the company pay for my courses and conferences?", "paraphrase", ("onboarding-all-2025",), "learning budget"),
    Q("Which days are people expected to be in the office?", "paraphrase", ("remote-all-2025",), "Tuesday and Thursday"),
    # Filtered: near-duplicate documents; the filter picks the right one. Run with and without it.
    Q("How many days of paid annual leave do I get?", "filtered", ("leave-uk-2025",), "24 days", {"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}),
    Q("How many days of paid annual leave do I get?", "filtered", ("leave-de-2025",), "30 days", {"region": "Germany"}),
    Q("What is the hotel cap per night?", "filtered", ("travel-us-2025",), "250 dollars", {"region": "US"}),
    Q("How many days a week can I work remotely?", "filtered", ("remote-de-2025",), "2 days a week", {"region": "Germany"}),
    Q("How many unused leave days can be carried over?", "filtered", ("leave-uk-2025",), "10 unused days", {"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}),
    Q("When does a medical certificate become necessary?", "filtered", ("leave-de-2025",), "fourth day", {"region": "Germany"}),
    # Not in the documents: the assistant must say there is not enough evidence.
    Q("What is the company dress code?", "out_of_scope"),
    Q("How much is the annual bonus?", "out_of_scope"),
    Q("When do stock options vest?", "out_of_scope"),
]

Writing corpus.py


In [3]:
from corpus import DOCS, QUESTIONS
print(len(DOCS), "documents,", len(QUESTIONS), "questions")
for d in DOCS[:3]:
    print(d.doc_id, "| region:", d.region, "| year:", d.year)

12 documents, 25 questions
leave-uk-2025 | region: UK | year: 2025
leave-uk-2024 | region: UK | year: 2024
leave-de-2025 | region: Germany | year: 2025


## Step 2: cut the documents into chunks
A long document is cut into small pieces so search can find the exact part. This is the same splitter as the project: about 400 characters per chunk, 60 characters shared between neighbours.

*Try:* change `chunk_size` to 200 and run the next steps again. How does the number of chunks change?

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)

chunks = []   # one dict per chunk: text and its labels
for d in DOCS:
    for i, text in enumerate(splitter.split_text(d.body)):
        chunks.append({"id": f"{d.doc_id}:{i+1}", "doc_id": d.doc_id, "title": d.title, "text": text,
                       "meta": {"department": d.department, "region": d.region, "year": d.year}})
print(len(chunks), "chunks")
print(chunks[0]["id"], "->", chunks[0]["text"][:120], "...")

17 chunks
leave-uk-2025:1 -> Every full-time UK employee gets 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per mon ...


## Step 3: turn every chunk into 384 numbers (an embedding)
The model reads a text and gives back a list of 384 numbers. Texts with similar meaning get similar numbers. The model runs on the CPU, no key needed.
Note: passages are embedded as they are, but a **question** gets a fixed sentence in front (BGE model rule). The project does the same.

In [5]:
import numpy as np
from fastembed import TextEmbedding
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

vectors = np.array(list(model.embed([c["text"] for c in chunks])), dtype=np.float32)
def embed_query(q):
    return np.array(next(iter(model.embed([QUERY_PREFIX + q]))), dtype=np.float32)

print("matrix:", vectors.shape, "-> one row of 384 numbers per chunk")
print("first 5 numbers of chunk 1:", vectors[0][:5].round(3))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

matrix: (17, 384) -> one row of 384 numbers per chunk
first 5 numbers of chunk 1: [-0.017 -0.011  0.021 -0.014  0.051]


## Step 4: how close are two meanings? (cosine similarity)
The score goes from 0 to 1. The vectors have length 1, so cosine similarity is just a dot product. In the project: similarity = 1 minus cosine distance.

In [6]:
def similarity(q, text):
    return float(embed_query(q) @ np.array(next(iter(model.embed([text]))), dtype=np.float32))

q = "After how many minutes does a company laptop lock itself?"
print("close meaning  :", round(similarity(q, "Company laptops lock automatically after 5 minutes."), 3))
print("different topic:", round(similarity(q, "Mentoring pairs meet once every 2 weeks for 45 minutes."), 3))

close meaning  : 0.813
different topic: 0.41


## Step 5: filters, then semantic search with Top K
A filter limits the search by label (region, year). This cell uses the **real** `matches()` function from `engine.py`. In the project the same filter becomes a safe SQL condition on a JSONB column.

In [7]:
def matches(meta: dict, flt: dict | None) -> bool:
    """Python check of a filter ($and, $or, $eq, $ne, $gt, $gte, $lt, $lte, $in). Used by the tests to verify what the database returned."""
    if not flt:
        return True
    for key, cond in flt.items():
        if key == "$and":
            if not all(matches(meta, c) for c in cond):
                return False
        elif key == "$or":
            if not any(matches(meta, c) for c in cond):
                return False
        else:
            v = meta.get(key)
            if not isinstance(cond, dict):
                cond = {"$eq": cond}
            for op, want in cond.items():
                ok = {"$eq": lambda: v == want, "$ne": lambda: v != want, "$gt": lambda: v is not None and v > want,
                      "$gte": lambda: v is not None and v >= want, "$lt": lambda: v is not None and v < want,
                      "$lte": lambda: v is not None and v <= want, "$in": lambda: v in want}.get(op)
                if ok is None:
                    raise ValueError(f"Unknown filter operator {op}")
                if not ok():
                    return False
    return True

Now the search. Compare the question with every chunk and take the **K best** that pass the filter. The real service does this with `embedding <=> query` in Postgres (with an HNSW index to avoid checking every row).

*Try:* change `k` to 1, 3 and 5 and look at the results.

In [8]:
def semantic(q, k=3, flt=None):
    qv = embed_query(q)
    sims = vectors @ qv
    order = [i for i in np.argsort(-sims) if matches(chunks[i]["meta"], flt)][:k]
    return [(chunks[i], float(sims[i])) for i in order]

for chunk, score in semantic("How often are performance reviews held?", k=3):
    print(round(score, 3), chunk["id"], "|", chunk["text"][:70].replace("\n", " "))

0.761 performance-all-2025:1 | Performance reviews take place twice a year, in June and in December. 
0.559 onboarding-all-2025:2 | Each employee has a learning budget of 1,500 euros per year for course
0.493 incident-all-2025:1 | A security incident is any lost device, leaked credential, suspected p


## Step 6: why a metadata filter matters
The question below has no single right answer without a filter: UK, Germany and an archived policy all talk about annual leave.

In [9]:
q = "How many days of paid annual leave do I get?"
print("WITHOUT filter:")
for c, s in semantic(q, k=3):
    print("  ", round(s, 3), c["id"], c["meta"]["region"], c["meta"]["year"])
print("WITH filter region = UK and year >= 2025:")
for c, s in semantic(q, k=3, flt={"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}):
    print("  ", round(s, 3), c["id"], c["meta"]["region"], c["meta"]["year"])

WITHOUT filter:
   0.865 leave-uk-2025:1 UK 2025
   0.824 leave-uk-2024:1 UK 2024
   0.797 leave-de-2025:1 Germany 2025
WITH filter region = UK and year >= 2025:
   0.865 leave-uk-2025:1 UK 2025
   0.744 leave-uk-2025:2 UK 2025
   0.649 holidays-uk-2026:1 UK 2026


## Step 7: keyword search and hybrid search
**Keyword search** finds exact words. Here it is simplified: it counts how many question words appear in a chunk (the project uses Postgres full text).
**Hybrid** merges the two ranked lists with *reciprocal rank fusion*: each chunk scores 1 / (60 + rank) from each list, and the scores are added. Chunks that rank well in either list rise. This is the real formula from `engine.py`.

In [10]:
import re
WORD = re.compile(r"[a-z0-9]+")
STOP = {"the", "a", "an", "of", "to", "is", "are", "do", "does", "how", "what", "when", "i", "my", "can", "or", "and", "in", "for", "after", "many", "much"}
def keyword(q, k=3, flt=None):
    qw = {w for w in WORD.findall(q.lower()) if w not in STOP}
    scored = []
    for i, c in enumerate(chunks):
        if not matches(c["meta"], flt):
            continue
        n = len(qw & set(WORD.findall(c["text"].lower())))
        if n:
            scored.append((n, i))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return [(chunks[i], float(n)) for n, i in scored[:k]]

def hybrid(q, k=3, flt=None):
    deep = max(k * 4, 20)
    fused, by_id = {}, {}
    for lst in (semantic(q, deep, flt), keyword(q, deep, flt)):
        for rank, (c, _) in enumerate(lst, start=1):
            fused[c["id"]] = fused.get(c["id"], 0.0) + 1.0 / (60 + rank)
            by_id.setdefault(c["id"], c)
    order = sorted(fused, key=lambda cid: -fused[cid])[:k]
    return [(by_id[cid], fused[cid]) for cid in order]

q = "Who signs off on a purchase above 500 euros?"
for name, fn in (("semantic", semantic), ("keyword", keyword), ("hybrid", hybrid)):
    print(name)
    for c, s in fn(q, 3):
        print("  ", round(s, 4), c["id"])

semantic
   0.7119 expense-all-2025:2
   0.6726 procurement-all-2025:1
   0.5326 expense-all-2025:1
keyword
   5.0 expense-all-2025:2
   4.0 procurement-all-2025:1
   3.0 holidays-uk-2026:1
hybrid
   0.0328 expense-all-2025:2
   0.0323 procurement-all-2025:1
   0.0313 holidays-uk-2026:1


## Step 8: measure it (Top K, search type and filters)
The 25 questions have known right documents. For each setting we count:
- **hit rate**: in how many questions the right document is in the top K,
- **MRR**: how high the right answer sits (1.0 means always first).

The real project reports: hit rate 81.8% at K=1 and 100% at K=5 for semantic search, and a filter raising MRR from 0.69 to 1.00. Your numbers here may differ a little, because the keyword search is simplified.

In [11]:
def evaluate(fn, k, qs, use_filter=False):
    hits, rr = 0, 0.0
    for qq in qs:
        res = fn(qq.question, k, qq.filter if use_filter else None)
        ranks = [i + 1 for i, (c, _) in enumerate(res) if c["doc_id"] in qq.docs]
        if ranks:
            hits += 1
            rr += 1.0 / ranks[0]
    n = len(qs)
    return round(100 * hits / n, 1), round(rr / n, 3)

answerable = [x for x in QUESTIONS if x.docs]
print("mode      K   hit rate %   MRR")
for name, fn in (("semantic", semantic), ("keyword", keyword), ("hybrid", hybrid)):
    for k in (1, 3, 5):
        h, m = evaluate(fn, k, answerable)
        print(f"{name:9} {k}   {h:9}   {m}")

filtered = [x for x in QUESTIONS if x.kind == "filtered"]
print()
print("Filtered questions (K=3, semantic):")
print("  without filter: hit %, MRR =", evaluate(semantic, 3, filtered, use_filter=False))
print("  with filter   : hit %, MRR =", evaluate(semantic, 3, filtered, use_filter=True))

mode      K   hit rate %   MRR
semantic  1        81.8   0.818
semantic  3        95.5   0.871
semantic  5       100.0   0.883
keyword   1        72.7   0.727
keyword   3        90.9   0.803
keyword   5        90.9   0.803
hybrid    1        77.3   0.773
hybrid    3        95.5   0.848
hybrid    5        95.5   0.848

Filtered questions (K=3, semantic):
  without filter: hit %, MRR = (100.0, 0.694)
  with filter   : hit %, MRR = (100.0, 1.0)


## Step 9: when the answer is not in the documents
Three questions are not in the documents at all (dress code, bonus, stock options). A good assistant should say "not enough evidence". One idea is a minimum similarity score (the *evidence gate*).

In [12]:
ans_best = [semantic(x.question, 1)[0][1] for x in answerable]
out_best = [semantic(x.question, 1)[0][1] for x in QUESTIONS if not x.docs]
print("lowest best-score of an answerable question :", round(min(ans_best), 3))
print("highest best-score of an unanswerable one   :", round(max(out_best), 3))
print("ranges overlap here:", min(ans_best) <= max(out_best))
print("The project's own evaluation measured 0.586 (lowest answerable) and 0.599 (highest unanswerable), which overlap.")
print("The numbers depend on how you measure, and the margin is thin, so a score threshold alone is a fragile gate.")

lowest best-score of an answerable question : 0.661
highest best-score of an unanswerable one   : 0.599
ranges overlap here: False
The project's own evaluation measured 0.586 (lowest answerable) and 0.599 (highest unanswerable), which overlap.
The numbers depend on how you measure, and the margin is thin, so a score threshold alone is a fragile gate.


## What to try next
1. Change `chunk_size` in Step 2 (for example 200 or 800) and run Steps 3 to 8 again. Which size gives the best hit rate?
2. Add your own document to `DOCS` (copy one `Doc(...)` line in `corpus.py`) and ask a question about it.
3. Swap the model name in Step 3 for another fastembed model and compare the table in Step 8.
4. Write down, in your own words, why the filter helps (Step 6) and why a score threshold alone is not enough (Step 9).

## Where this is in the real code
- Chunking, embedding, storing: `vector-lab/app/engine.py`, method `ingest`.
- Semantic and keyword search, hybrid merge, filters: `engine.py`, methods `_semantic`, `_keyword`, `search`, and `filter_sql`.
- The screen in the portal: **Practice lab, Vector search**.